# JsonOutputParser
> LLM이 출력한 텍스트를 Json형태로 파싱하는 출력 파서<br>
> LLM이 생성한 문자열 결과를 자동으로 dict로 변환해주는 도구

### Parser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

# JsonOutputParser 초기화
# 모델의 응답을 JSON 형태로 파싱하기 위한 출력 파서 생성
json_parser = JsonOutputParser()

In [2]:
# 모델이 JSON 형식으로 답하도록 안내하는 출력 지침 생성
format_instructions = json_parser.get_format_instructions()

print("출력 형식 지침: ")
print(format_instructions)

출력 형식 지침: 
Return a JSON object.


## Prompt

In [ ]:
from langchain_core.prompts import PromptTemplate

# 한국 여행지 정보를 위한 프롬프트 템플릿
travel_prompt = PromptTemplate(
    template="""
    다음 한국 여행지에 대한 정보를 JSON 형식으로 제공해주세요.

    여행지: {destination}

    다음 필드들을 포함해주세요:
    - name: 여행지 이름
    - location:위치 (시/도)
    - attractions: 주요 관광명소(배열)
    - best_season: 최적 방문시기
    - budget: 예상 비용 (1일 기준)
    - specialities: 지역 특산물/음식 (배열)

    {format_instructions}
    """,
    input_variables=["destination"],
    partial_variables={"format_instructions": json_parser.get_format_instructions()}
)

In [11]:
# travel_prompt의 템플릿에 입력해야하는 변수 확인
# 템플릿 실행 시 사용자가 넣어야 하는 입력 변수 확인
travel_prompt.input_variables

['destination']

In [12]:
# travel_prompt의 템플릿 형식 출력
# 현재 프롬프트 템플릿 내용을 출력
print(travel_prompt.template)


    다음 한국 여행지에 대한 정보를 JSON 형식으로 제공해주세요.

    여행지: {destination}

    다음 필드들을 포함해주세요:
    - name: 여행지 이름
    - locationL:위치 (시/도)
    - attractions: 주요 관광명소(배열)
    - best_season: 최적 방문시기
    - budget: 예상 비용 (1일 기준)
    - specialities: 지역 특산물/음식 (배열)

    {format_instructions}
    


## Model
### OpenAI API Key 발급

In [13]:
from dotenv import load_dotenv

# .env 파일에서 환경변수 로드
load_dotenv()

True

In [14]:
import os 

# API 키 확인
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("OpenAI API 키가 설정되었습니다. (GPT 모델 사용)")
else:
    print("OpenAI API 키가 없습니다.")

OpenAI API 키가 없습니다.


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    # 사용할 OpenAI 채팅 모델 지정
    model="gpt-5-nano",             # gpt-5 모델 사용
    reasoning_effort="high",        # 논리성 강화, 추론에 더 많은 계산/노력을 사용하도록 설정
)

## Chain with Parser
> 사용자 입력 -> 프롬프트 구성 -> LLM 호출 -> 출력 파싱 -> 결과 반환

In [15]:
# 프롬프트, 모델, Json 파서를 연결하여 체인 생성  
travel_chain = travel_prompt | model | json_parser

NameError: name 'model' is not defined

In [ ]:
# destination에 "제주도" 입력해 체인 실행
# 결과가 JSON 파싱 후 파이썬 딕셔너리 형태로 반환됨
jeju_info = travel_chain.invoke({"destination":"제주도"})

In [ ]:
# 모델에서 받은 제주에 대한 정보를 출력
# 결과는 Json 문자열이 아니라 파이썬 딕셔너리 형태
print("제주도 여행 정보:")
jeju_info

In [ ]:
# 딕셔너리에서 name이 키값인 값
jeju_info['name']

In [ ]:
# 딕셔너리에서 specialities이 키값인 
jeju_info['specialities']